# Running evaluations on Isambard

Evaluate **gpt-oss-120b** with the UK AISI
[Inspect](https://inspect.aisi.org.uk/) framework, against a vLLM server
running on an Isambard GH200 node.

**Start the server first**, from a login node:

```bash
bash serve_vllm.sh
```

It serves the model as an OpenAI-compatible endpoint and writes its URL to
`endpoint.txt`. Select the **`.venv-eval`** kernel for this notebook (the
serving and eval stacks are separate venvs -- see the README). Everything below talks HTTP, so the notebook never touches a
GPU -- the ~4-minute model load is paid once by the server, and you iterate
against it interactively.

Based on the official
[distributed-inference tutorial](https://docs.isambard.ac.uk/user-documentation/tutorials/distributed-inference/).


## 1. Connect to the server

In [ ]:
import os, pathlib, urllib.request, json, logging, warnings

# Inspect runs an async event loop; under Jupyter its teardown emits harmless
# noise ("Task was destroyed", "coroutine was never awaited"). Quieten it.
logging.getLogger("asyncio").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore", message="coroutine .* was never awaited")

# serve_vllm.sh writes this when the server starts.
endpoint_file = pathlib.Path("endpoint.txt")
if endpoint_file.exists():
    ENDPOINT = endpoint_file.read_text().strip()
else:
    # Fall back to a manual value, e.g. if you started the server yourself.
    ENDPOINT = os.environ.get("ISAMBARD_BASE_URL", "http://localhost:8000/v1")

MODEL_NAME = os.environ.get("SERVED_NAME", "gpt-oss-120b")

# Inspect resolves `openai-api/<service>/<model>` by reading <SERVICE>_BASE_URL
# and <SERVICE>_API_KEY. The key is unused by vLLM but must be non-empty.
os.environ["ISAMBARD_BASE_URL"] = ENDPOINT
os.environ["ISAMBARD_API_KEY"] = "dummy"

print(f"endpoint : {ENDPOINT}")
print(f"model    : {MODEL_NAME}")

with urllib.request.urlopen(ENDPOINT + "/models", timeout=30) as r:
    served = [m["id"] for m in json.load(r)["data"]]
print(f"serving  : {served}")
assert MODEL_NAME in served, f"{MODEL_NAME} not served; found {served}"

## 2. A first evaluation

`inspect_evals` ships the standard benchmarks. GSM8K is grade-school maths
word problems — small, fast, and a reasonable smoke test that the whole chain
works.

`--limit` keeps this interactive. Drop it for a real run, but note the full
GSM8K test split is 1319 problems.

In [ ]:
from inspect_ai import eval as inspect_eval
from inspect_evals.gsm8k import gsm8k

MODEL = f"openai-api/isambard/{MODEL_NAME}"

logs = inspect_eval(
    gsm8k(),
    model=MODEL,
    limit=10,             # 10 samples, not all 1319
    max_connections=10,   # vLLM batches concurrent requests efficiently
)
log = logs[0]
print(f"\nstatus  : {log.status}")
print(f"samples : {len(log.samples or [])}")

### The headline numbers

In [ ]:
# A scorer reports one or more metrics; walk them rather than assuming names.
for score in (log.results.scores if log.results else []):
    print(f"scorer: {score.name}")
    for metric_name, metric in score.metrics.items():
        print(f"  {metric_name:10s} {metric.value:.3f}")

print(f"\ntokens: {log.stats.model_usage}")


## 3. Looking at individual answers

The aggregate number is rarely the interesting part. Inspect keeps every
sample — the prompt, the model's full output, and why the scorer accepted or
rejected it. This is where you find out that your eval is measuring formatting
rather than reasoning.

In [ ]:
for s in (log.samples or [])[:3]:
    verdict = list(s.scores.values())[0] if s.scores else None
    print("=" * 70)
    print("Q:", str(s.input)[:200].replace("\n", " "))
    print("-" * 70)
    print("A:", str(s.output.completion)[:400].replace("\n", " "))
    print(f"--> target={s.target}  score={verdict.value if verdict else '?'}")

## 4. Where the answers went wrong

Filter to the failures. With `limit=10` there may well be none — raise the
limit if you want something to look at.

In [ ]:
def is_incorrect(sample):
    if not sample.scores:
        return False
    return list(sample.scores.values())[0].value not in ("C", 1, 1.0, True)

wrong = [s for s in (log.samples or []) if is_incorrect(s)]
print(f"{len(wrong)} incorrect of {len(log.samples or [])}\n")

for s in wrong[:2]:
    print("=" * 70)
    print("Q:", str(s.input)[:200].replace("\n", " "))
    print("model :", str(s.output.completion)[-250:].replace("\n", " "))
    print("target:", s.target)

## 5. Comparing configurations

Because the model sits behind an endpoint, comparing generation settings costs
nothing but tokens — no reloading, no requeueing. Here, does temperature change
the score?

This is the pattern that makes an interactive session worth holding: the 5
minutes of model load is paid once, and you iterate against it.

In [ ]:
from inspect_ai.model import GenerateConfig

# eval() forwards unknown kwargs into GenerateConfig, so temperature/top_p/
# max_tokens can be passed straight through.
results = {}
for temperature in (0.0, 1.0):
    lg = inspect_eval(
        gsm8k(),
        model=MODEL,
        limit=10,
        display="plain",
        max_connections=10,
        temperature=temperature,
    )[0]
    acc = lg.results.scores[0].metrics["accuracy"].value if lg.results else float("nan")
    results[temperature] = acc
    print(f"temperature={temperature}  accuracy={acc:.3f}")

print("\nsummary:", results)


## 6. Where the logs go

Every run writes a `.eval` log under `logs/`. They are the durable record —
full transcripts, scores, token counts, and the exact config — and they are
what you should keep rather than the notebook output.

Browse them in a rich UI with:

```bash
uv run inspect view --log-dir logs
```

which serves a local web app; forward the port through your VS Code tunnel to
open it in a browser.

In [ ]:
import pathlib
for p in sorted(pathlib.Path("logs").glob("*.eval"))[-5:]:
    print(f"  {p.name}  ({p.stat().st_size/1024:.0f} KB)")

---

## Next

- **Other benchmarks:** `inspect_evals` ships MMLU, ARC, HumanEval and many
  more -- `from inspect_evals.mmlu import mmlu` and swap it in above.
- **Your own eval:** an Inspect `Task` is a dataset, a solver and a scorer.
  Writing one is the natural next step once a benchmark runs.
- **Stop the server (Ctrl-C)** when you are done -- it holds four GPUs, and
  the cluster runs near capacity.
